In [25]:
%%writefile /content/train_spatialglue.py
# ============================================================
# SpatialGlue benchmark runner
# 运行方式: /usr/local/miniconda/envs/sg/bin/python train_spatialglue.py
# ============================================================

import sys
print("Python:", sys.version, flush=True)
assert sys.version_info[:2] == (3, 8), "必须用 sg 环境的 python 运行"

import gc, argparse
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import torch

from SpatialGlue.preprocess import construct_neighbor_graph
from SpatialGlue.SpatialGlue_pyG import Train_SpatialGlue

DATA_ROOT = Path("/content/drive/MyDrive/SMGC-data")
OUT_ROOT  = Path("/content/drive/MyDrive/table2_benchmark/baseline_predictions")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "HLN-A1": {"rel": "Human_Lymph_Nodes/A1", "rna": "adata_RNA.h5ad", "other": "adata_ADT.h5ad",
               "label_key": "Spatial_Label", "n_clusters": 10, "spatial_k": 3},
    "HLN-D1": {"rel": "Human_Lymph_Nodes/D1", "rna": "adata_RNA.h5ad", "other": "adata_ADT.h5ad",
               "label_key": "Spatial_Label", "n_clusters": 11, "spatial_k": 3},
    "E18.5":  {"rel": "E18.5_mouse_brain", "rna": "adata_RNA.h5ad", "other": "adata_ATAC.h5ad",
               "label_key": "Combined_Clusters", "n_clusters": 14, "spatial_k": 3},
    "S2-E15": {"rel": "Mouse_Embryos_S2/E15", "rna": "adata_RNA.h5ad", "other": "adata_ATAC.h5ad",
               "label_key": "Spatial_Label", "n_clusters": 15, "spatial_k": 3},
    "S2-E18": {"rel": "Mouse_Embryos_S2/E18", "rna": "adata_RNA.h5ad", "other": "adata_ATAC.h5ad",
               "label_key": "Spatial_Label", "n_clusters": 16, "spatial_k": 5},
}


def load_one(dataset, n_pca=100):
    s = DATASETS[dataset]
    base = DATA_ROOT / s["rel"]
    a1 = ad.read_h5ad(base / s["rna"])
    a2 = ad.read_h5ad(base / s["other"])
    assert list(a1.obs_names) == list(a2.obs_names), f"{dataset}: spot 顺序不一致"

    def pca_whiten(X, n):
        X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
        X = X.astype(np.float32)
        n_comp = min(n, X.shape[1], X.shape[0])
        return PCA(n_components=n_comp, whiten=True, random_state=0).fit_transform(X).astype(np.float32)

    X1 = pca_whiten(a1.X, n_pca)
    X2 = pca_whiten(a2.X, n_pca)

    if "spatial" in a1.obsm:
        sp = np.asarray(a1.obsm["spatial"], dtype=np.float32)
    elif "array_row" in a1.obs.columns and "array_col" in a1.obs.columns:
        sp = np.stack([a1.obs["array_row"].values, a1.obs["array_col"].values], axis=1).astype(np.float32)
    else:
        raise ValueError(f"{dataset}: 没找到 spatial 坐标")

    return X1, X2, sp, s["spatial_k"], s["n_clusters"], np.asarray(a1.obs_names)


def run_one(X1, X2, sp, spatial_k, n_clusters, seed, device):
    torch.manual_seed(seed)
    np.random.seed(seed)

    a1 = ad.AnnData(X=X1); a2 = ad.AnnData(X=X2)
    a1.obsm["feat"] = X1.copy(); a2.obsm["feat"] = X2.copy()
    a1.obsm["spatial"] = sp.copy(); a2.obsm["spatial"] = sp.copy()

    data = construct_neighbor_graph(a1, a2, datatype='SPOTS', n_neighbors=spatial_k)

    # ✅ 只传 dim_output，模态维度自动从数据推断
    model = Train_SpatialGlue(
        data,
        datatype='SPOTS',
        device=device,
        random_seed=seed,
        learning_rate=1e-4,
        weight_decay=0.0,
        epochs=600,           # SPOTS 时会强制 600
        dim_output=64,
    )
    out = model.train()
    emb = out['SpatialGlue']

    return KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit_predict(emb)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", default=None, help="只跑某个数据集；不指定则跑全部")
    p.add_argument("--seeds", default="0-9", help="形如 '0-9' 或 '0'")
    args = p.parse_args()

    if "-" in args.seeds:
        a, b = args.seeds.split("-")
        seeds = list(range(int(a), int(b) + 1))
    else:
        seeds = [int(args.seeds)]

    datasets = [args.dataset] if args.dataset else list(DATASETS.keys())

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("设备:", device, flush=True)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0), flush=True)

    for dataset in datasets:
        print("\n" + "=" * 80, flush=True)
        print("Loading dataset:", dataset, flush=True)
        X1, X2, sp, spatial_k, n_clusters, spot_ids = load_one(dataset)
        print(f"  X1={X1.shape}, X2={X2.shape}, spatial={sp.shape}, k={spatial_k}, n_clusters={n_clusters}", flush=True)

        for seed in seeds:
            out_csv = OUT_ROOT / f"SpatialGlue_{dataset}_seed{seed:02d}.csv"
            if out_csv.exists():
                print(f"  [skip] {out_csv.name} 已存在", flush=True)
                continue

            print(f"  Running seed {seed} ...", end=" ", flush=True)
            try:
                y_pred = run_one(X1, X2, sp, spatial_k, n_clusters, seed, device)
                pd.DataFrame({
                    "spot_id": spot_ids.astype(str),
                    "pred_label": y_pred,
                }).to_csv(out_csv, index=False)
                print(f"✅ saved -> {out_csv.name}", flush=True)
            except Exception as e:
                print(f"❌ FAILED: {type(e).__name__}: {e}", flush=True)
            finally:
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()

    print("\n全部完成。输出目录:", OUT_ROOT, flush=True)


if __name__ == "__main__":
    main()

Overwriting /content/train_spatialglue.py


In [26]:
!/usr/local/miniconda/envs/sg/bin/python /content/train_spatialglue.py --dataset HLN-A1 --seeds 0

Python: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
设备: cuda
GPU: Tesla T4

Loading dataset: HLN-A1
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
  X1=(3484, 100), X2=(3484, 31), spatial=(3484, 2), k=3, n_clusters=10
  Running seed 0 ... /usr/local/miniconda/envs/sg/lib/python3.8/site-packages/SpatialGlue/preprocess.py:132: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:605.)
  return torch.sparse.FloatTensor(

In [27]:
from pathlib import Path
OUT = Path("/content/drive/MyDrive/table2_benchmark/baseline_predictions")
files = sorted(OUT.glob("SpatialGlue_*.csv"))
print(f"共 {len(files)} 个文件 (期望 50)")
for f in files:
    print(" ", f.name)

共 1 个文件 (期望 50)
  SpatialGlue_HLN-A1_seed00.csv


In [28]:
!/usr/local/miniconda/envs/sg/bin/python /content/train_spatialglue.py

Python: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
设备: cuda
GPU: Tesla T4

Loading dataset: HLN-A1
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
  X1=(3484, 100), X2=(3484, 31), spatial=(3484, 2), k=3, n_clusters=10
  [skip] SpatialGlue_HLN-A1_seed00.csv 已存在
  Running seed 1 ... /usr/local/miniconda/envs/sg/lib/python3.8/site-packages/SpatialGlue/preprocess.py:132: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.c

In [29]:
from pathlib import Path
OUT = Path("/content/drive/MyDrive/table2_benchmark/baseline_predictions")
files = sorted(OUT.glob("SpatialGlue_*.csv"))
print(f"共 {len(files)} 个文件 (期望 50)")
for f in files:
    print(" ", f.name)

共 50 个文件 (期望 50)
  SpatialGlue_E18.5_seed00.csv
  SpatialGlue_E18.5_seed01.csv
  SpatialGlue_E18.5_seed02.csv
  SpatialGlue_E18.5_seed03.csv
  SpatialGlue_E18.5_seed04.csv
  SpatialGlue_E18.5_seed05.csv
  SpatialGlue_E18.5_seed06.csv
  SpatialGlue_E18.5_seed07.csv
  SpatialGlue_E18.5_seed08.csv
  SpatialGlue_E18.5_seed09.csv
  SpatialGlue_HLN-A1_seed00.csv
  SpatialGlue_HLN-A1_seed01.csv
  SpatialGlue_HLN-A1_seed02.csv
  SpatialGlue_HLN-A1_seed03.csv
  SpatialGlue_HLN-A1_seed04.csv
  SpatialGlue_HLN-A1_seed05.csv
  SpatialGlue_HLN-A1_seed06.csv
  SpatialGlue_HLN-A1_seed07.csv
  SpatialGlue_HLN-A1_seed08.csv
  SpatialGlue_HLN-A1_seed09.csv
  SpatialGlue_HLN-D1_seed00.csv
  SpatialGlue_HLN-D1_seed01.csv
  SpatialGlue_HLN-D1_seed02.csv
  SpatialGlue_HLN-D1_seed03.csv
  SpatialGlue_HLN-D1_seed04.csv
  SpatialGlue_HLN-D1_seed05.csv
  SpatialGlue_HLN-D1_seed06.csv
  SpatialGlue_HLN-D1_seed07.csv
  SpatialGlue_HLN-D1_seed08.csv
  SpatialGlue_HLN-D1_seed09.csv
  SpatialGlue_S2-E15_seed00.csv
 

In [31]:
%%writefile /content/summarize_spatialglue.py
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

DATA_ROOT = Path("/content/drive/MyDrive/SMGC-data")
OUT_ROOT  = Path("/content/drive/MyDrive/table2_benchmark/baseline_predictions")

DATASETS = {
    "HLN-A1": {"rel": "Human_Lymph_Nodes/A1", "rna": "adata_RNA.h5ad",
               "label_key": "Spatial_Label"},
    "HLN-D1": {"rel": "Human_Lymph_Nodes/D1", "rna": "adata_RNA.h5ad",
               "label_key": "Spatial_Label"},
    "E18.5":  {"rel": "E18.5_mouse_brain", "rna": "adata_RNA.h5ad",
               "label_key": "Combined_Clusters"},
    "S2-E15": {"rel": "Mouse_Embryos_S2/E15", "rna": "adata_RNA.h5ad",
               "label_key": "Spatial_Label"},
    "S2-E18": {"rel": "Mouse_Embryos_S2/E18", "rna": "adata_RNA.h5ad",
               "label_key": "Spatial_Label"},
}

rows = []
for ds, s in DATASETS.items():
    a = ad.read_h5ad(DATA_ROOT / s["rel"] / s["rna"], backed="r")
    gt = pd.DataFrame({
        "spot_id": a.obs_names.astype(str),
        "gt": a.obs[s["label_key"]].astype(str).to_numpy(),
    })

    aris, nmis = [], []
    for seed in range(10):
        pred_csv = OUT_ROOT / f"SpatialGlue_{ds}_seed{seed:02d}.csv"
        pred = pd.read_csv(pred_csv)
        merged = gt.merge(pred, on="spot_id", how="left", validate="one_to_one")
        assert merged["pred_label"].notna().all(), f"{pred_csv} 有缺失"
        aris.append(adjusted_rand_score(merged["gt"], merged["pred_label"]))
        nmis.append(normalized_mutual_info_score(merged["gt"], merged["pred_label"]))

    rows.append({
        "Dataset": ds,
        "ARI_mean": np.mean(aris),
        "ARI_std":  np.std(aris, ddof=1),
        "NMI_mean": np.mean(nmis),
        "NMI_std":  np.std(nmis, ddof=1),
        "n":        len(aris),
    })

df = pd.DataFrame(rows)
df["ARI"] = df.apply(lambda r: f"{r.ARI_mean:.4f} ± {r.ARI_std:.4f}", axis=1)
df["NMI"] = df.apply(lambda r: f"{r.NMI_mean:.4f} ± {r.NMI_std:.4f}", axis=1)

print("SpatialGlue on 5 datasets (10 seeds):")
print(df[["Dataset", "ARI", "NMI", "n"]].to_string(index=False))

out_csv = Path("/content/drive/MyDrive/table2_benchmark/SpatialGlue_summary.csv")
df.to_csv(out_csv, index=False)
print("\n✅ 已保存:", out_csv)

Writing /content/summarize_spatialglue.py


In [32]:
!/usr/local/miniconda/envs/sg/bin/python /content/summarize_spatialglue.py

/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/miniconda/envs/sg/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
SpatialGlue on 5 datasets (10 seeds):
Dataset             ARI             NMI  n
 HLN-A1 0.1332 ± 0.0152 0.2595 ± 0.

In [33]:
%%writefile /content/build_table2.py
import pandas as pd
from pathlib import Path

SG_PATH      = Path("/content/drive/MyDrive/table2_benchmark/SpatialGlue_summary.csv")
SPAMGCL_PATH = Path("/content/SpaMGCL_summary.csv")
OUT_PATH     = Path("/content/drive/MyDrive/table2_benchmark/Table2_ready.csv")

assert SG_PATH.exists(), f"找不到 {SG_PATH}"
assert SPAMGCL_PATH.exists(), f"找不到 {SPAMGCL_PATH}，请先从 Kaggle 下载并上传到 /content/"

sg      = pd.read_csv(SG_PATH)
spamgcl = pd.read_csv(SPAMGCL_PATH)

print("=" * 70)
print("SpatialGlue 列名:", list(sg.columns))
print(sg.head())
print("\n" + "=" * 70)
print("SpaMGCL 列名:", list(spamgcl.columns))
print(spamgcl.head())
print("=" * 70)

sg = sg.rename(columns={"Dataset": "dataset", "n": "n_sg"})

method_order  = ["SpaMGCL", "SpatialGlue"]
dataset_order = ["HLN-A1", "HLN-D1", "E18.5", "S2-E15", "S2-E18"]

rows = []
for _, r in spamgcl.iterrows():
    rows.append({"method": "SpaMGCL", "dataset": r["dataset"], "n": r["n"],
                 "ARI_mean": r["ARI_mean"], "ARI_std": r["ARI_std"],
                 "NMI_mean": r["NMI_mean"], "NMI_std": r["NMI_std"]})
for _, r in sg.iterrows():
    rows.append({"method": "SpatialGlue", "dataset": r["dataset"], "n": r["n_sg"],
                 "ARI_mean": r["ARI_mean"], "ARI_std": r["ARI_std"],
                 "NMI_mean": r["NMI_mean"], "NMI_std": r["NMI_std"]})

long = pd.DataFrame(rows)
long["ARI"] = long.apply(lambda r: f"{r['ARI_mean']:.4f} ± {r['ARI_std']:.4f}", axis=1)
long["NMI"] = long.apply(lambda r: f"{r['NMI_mean']:.4f} ± {r['NMI_std']:.4f}", axis=1)
long["dataset"] = pd.Categorical(long["dataset"], categories=dataset_order, ordered=True)

table = []
for ds in dataset_order:
    sub = long[long["dataset"] == ds]
    for metric in ["ARI", "NMI"]:
        row = {"Dataset": ds, "Metric": metric}
        for m in method_order:
            v = sub[sub["method"] == m][metric]
            row[m] = v.iloc[0] if len(v) else "PENDING"
        table.append(row)

t2 = pd.DataFrame(table)
print("\n" + "=" * 70)
print("Table 2 (10 seeds, mean ± std)")
print("=" * 70)
print(t2.to_string(index=False))

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
t2.to_csv(OUT_PATH, index=False)
long.to_csv(OUT_PATH.with_name("Table2_long.csv"), index=False)
print("\n✅ 已保存:", OUT_PATH)

Writing /content/build_table2.py


In [36]:
import zipfile
from pathlib import Path

SRC = Path("/content/drive/MyDrive/table2_benchmark")
ZIP_PATH = "/content/drive/MyDrive/table2_benchmark_full.zip"

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in SRC.rglob("*"):
        if f.is_file() and f.name != "table2_benchmark_full.zip":
            zf.write(f, arcname=f.relative_to(SRC.parent))

import os
print("打包完成:", ZIP_PATH)
print("大小:", round(os.path.getsize(ZIP_PATH)/1024/1024, 2), "MB")
print("文件数:", len(zipfile.ZipFile(ZIP_PATH).namelist()))

打包完成: /content/drive/MyDrive/table2_benchmark_full.zip
大小: 0.63 MB
文件数: 51


In [37]:
from google.colab import files
files.download("/content/drive/MyDrive/table2_benchmark_full.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>